# Noisy Weights — Convergence Analysis

Analyses convergence behaviour when training from different random weight perturbations at **noise level 0.4**. Each run starts from a different random seed. The question: does the network reliably learn to match the teacher?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from connectome_snns.utils.reproducibility import load_experiment_config
from connectome_snns.visualization import use_project_style, plot_spike_trains
from connectome_snns.visualization.scaling_factors import SF_PATHWAYS, plot_sf_trajectories
from connectome_snns.analysis import interleave_spike_trains

use_project_style()

In [ ]:
# Load inference results at noise=0.4 (scaling factors at target)
inference_config = load_experiment_config("../inference/experiment.toml")
inference_dir = inference_config["output_dir"] / "noise-0.40"

HAS_INFERENCE = inference_dir.exists()

if HAS_INFERENCE:
    inference_metrics = pd.read_csv(inference_dir / "training_metrics.csv")

    # Extract inference values (baseline with SF at target)
    inference_loss = float(inference_metrics["loss"].iloc[0])
    inference_fr_exc = float(inference_metrics["firing_rate/student_exc_mean"].iloc[0])
    inference_fr_inh = float(inference_metrics["firing_rate/student_inh_mean"].iloc[0])

    # Teacher values
    teacher_fr_exc = float(inference_metrics["firing_rate/teacher_exc_mean"].iloc[0])
    teacher_fr_inh = float(inference_metrics["firing_rate/teacher_inh_mean"].iloc[0])

    print("Correct Student at noise=0.4 (SF at target):")
    print(f"  Loss: {inference_loss:.4f}")
    print(
        f"  Student FR - Exc: {inference_fr_exc:.2f} Hz, Inh: {inference_fr_inh:.2f} Hz"
    )
    print(f"  Teacher FR - Exc: {teacher_fr_exc:.2f} Hz, Inh: {teacher_fr_inh:.2f} Hz")
else:
    print("SKIPPED: inference data not found at", inference_dir)

In [ ]:
# Load convergence data from multiple seeds (noise=0.4)
convergence_config = load_experiment_config("experiment.toml")
convergence_dir = convergence_config["output_dir"]

HAS_CONVERGENCE = convergence_dir.exists()

if HAS_CONVERGENCE:
    seed_dirs = sorted(
        [
            d
            for d in convergence_dir.iterdir()
            if d.is_dir() and d.name.startswith("seed-")
        ]
    )
    print(f"Found {len(seed_dirs)} convergence runs")

    # Load trajectories
    trajectories = {}
    for seed_dir in seed_dirs:
        seed = int(seed_dir.name.split("-")[1])
        df = pd.read_csv(seed_dir / "training_metrics.csv")
        traj = {
            "epochs": df["epoch"].values,
            "total_loss": df["total_loss"].values,
            "student_firing_rate_exc": df["firing_rate/student_excitatory_mean"].values,
            "student_firing_rate_inh": df["firing_rate/student_inhibitory_mean"].values,
        }
        for key, _ in SF_PATHWAYS:
            traj[key] = df[f"scaling_factors/{key}_value"].values
        trajectories[seed] = traj

    seeds = sorted(trajectories.keys())
    epochs_ref = trajectories[seeds[0]]["epochs"]
    print(f"Seeds: {seeds}")
    print(f"Epochs: 0 to {int(epochs_ref[-1] / 50)}")
else:
    print("SKIPPED: convergence-check data not found at", convergence_dir)

In [ ]:
if not HAS_CONVERGENCE:
    print("SKIPPED: no convergence data")
else:
    # Build metrics dict keyed by seed for plot_sf_trajectories
    sf_metrics = {}
    for s in seeds:
        sf_metrics[s] = pd.DataFrame(
            {
                "epoch": trajectories[s]["epochs"] / 50,
                **{
                    f"scaling_factors/{key}_value": trajectories[s][key]
                    for key, _ in SF_PATHWAYS
                },
            }
        )

    fig = plot_sf_trajectories(
        sf_metrics,
        mean_std=True,
        suptitle="Scaling Factor Convergence (Mean ± Std across Seeds)",
    )
    plt.show()

In [ ]:
if not (HAS_CONVERGENCE and HAS_INFERENCE):
    print("SKIPPED: need both convergence and inference data")
else:
    from connectome_snns.visualization.training_curves import plot_loss_mean_std

    # Build metrics dict for loss plotting
    loss_metrics = {}
    for s in seeds:
        loss_metrics[s] = pd.DataFrame(
            {
                "epoch": trajectories[s]["epochs"] / 50,
                "total_loss": trajectories[s]["total_loss"],
            }
        )

    fig = plot_loss_mean_std(
        loss_metrics,
        title="Loss Trajectory",
        reference_lines={f"Correct Student: {inference_loss:.2f}": inference_loss},
    )
    plt.show()

In [ ]:
if not (HAS_CONVERGENCE and HAS_INFERENCE):
    print("SKIPPED: need both convergence and inference data")
else:
    # Plot: Population firing rate trajectories by cell type
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    fr_keys = [
        ("student_firing_rate_exc", "Excitatory", teacher_fr_exc, inference_fr_exc),
        ("student_firing_rate_inh", "Inhibitory", teacher_fr_inh, inference_fr_inh),
    ]

    for ax, (key, title, teacher_val, inference_val) in zip(axes, fr_keys):
        fr_all = np.array([trajectories[s][key] for s in seeds])
        mean_fr = fr_all.mean(axis=0)
        std_fr = fr_all.std(axis=0)

        ax.fill_between(
            epochs_ref / 50, mean_fr - std_fr, mean_fr + std_fr, alpha=0.3, color="C0"
        )
        ax.plot(
            epochs_ref / 50, mean_fr, color="C0", linewidth=2, label="Trained Student"
        )
        ax.axhline(
            y=teacher_val,
            color="C1",
            linestyle="--",
            linewidth=2,
            label=f"Teacher: {teacher_val:.2f} Hz",
        )
        ax.axhline(
            y=inference_val,
            color="C2",
            linestyle=":",
            linewidth=2,
            label=f"Correct Student: {inference_val:.2f} Hz",
        )

        ax.set_xlabel("Epoch")
        ax.set_ylabel("Population Firing Rate (Hz)")
        ax.set_title(title)
        ax.set_ylim(0, None)
        ax.legend(loc="upper right")

    plt.suptitle(
        "Population Firing Rate Trajectories (Mean ± Std across seeds)",
        fontweight="bold",
    )
    plt.tight_layout()
    plt.show()

In [ ]:
if not HAS_CONVERGENCE:
    print("SKIPPED: no convergence data")
else:
    # Convergence summary: final values
    print("Convergence Summary (Final Epoch, Noise = 0.4):")
    print("=" * 80)

    final_loss = np.array([trajectories[s]["total_loss"][-1] for s in seeds])
    final_fr_exc = np.array(
        [trajectories[s]["student_firing_rate_exc"][-1] for s in seeds]
    )
    final_fr_inh = np.array(
        [trajectories[s]["student_firing_rate_inh"][-1] for s in seeds]
    )

    inf_loss_str = f"{inference_loss:.4f}" if HAS_INFERENCE else "N/A"
    inf_fr_exc_str = f"{inference_fr_exc:.2f}" if HAS_INFERENCE else "N/A"
    inf_fr_inh_str = f"{inference_fr_inh:.2f}" if HAS_INFERENCE else "N/A"
    teacher_exc_str = f"{teacher_fr_exc:.2f}" if HAS_INFERENCE else "N/A"
    teacher_inh_str = f"{teacher_fr_inh:.2f}" if HAS_INFERENCE else "N/A"

    print(
        f"\n{'Metric':<25} {'Trained Student (mean±std)':>28} {'Correct Student':>18} {'Teacher':>12}"
    )
    print("-" * 85)
    print(
        f"{'Loss':<25} {final_loss.mean():>10.4f} ± {final_loss.std():.4f} {inf_loss_str:>18} {'':>12}"
    )
    print(
        f"{'FR Exc (Hz)':<25} {final_fr_exc.mean():>10.2f} ± {final_fr_exc.std():.2f} {inf_fr_exc_str:>18} {teacher_exc_str:>12}"
    )
    print(
        f"{'FR Inh (Hz)':<25} {final_fr_inh.mean():>10.2f} ± {final_fr_inh.std():.2f} {inf_fr_inh_str:>18} {teacher_inh_str:>12}"
    )

    print("\nScaling Factors (target = 1.0):")
    for key, title in SF_PATHWAYS:
        final_sf = np.array([trajectories[s][key][-1] for s in seeds])
        print(f"  {title:<20} {final_sf.mean():>8.4f} ± {final_sf.std():.4f}")

### Spike Train Visualization

Visualizes learned spike patterns vs the teacher for example seed (seed-42).

In [ ]:
if not HAS_CONVERGENCE:
    print("SKIPPED: no convergence data")
else:
    example_seed = 42
    example_seed_dir = convergence_dir / f"seed-{example_seed}"
    plot_data_path = example_seed_dir / "final_state" / "plot_data.npz"

    print(f"Loading plot data from: {plot_data_path}")
    plot_data = np.load(plot_data_path)

    student_spikes = plot_data["student_spikes"]  # (time, n_neurons)
    teacher_spikes = plot_data["teacher_spikes"]  # (time, n_neurons)
    dt = float(plot_data["dt"])
    cell_type_indices = plot_data["cell_type_indices"]
    n_neurons = int(plot_data["n_neurons"])

    print(f"Loaded data: {n_neurons} neurons, {student_spikes.shape[0]} timesteps")
    print(f"Duration: {student_spikes.shape[0] * dt / 1000:.2f} seconds")

In [ ]:
if not HAS_CONVERGENCE:
    print("SKIPPED: no convergence data")
else:
    interleaved, ct_idx = interleave_spike_trains(teacher_spikes, student_spikes)

    fig = plot_spike_trains(
        spikes=interleaved,
        dt=dt,
        cell_type_indices=ct_idx,
        cell_type_names=["Teacher", "Trained Student"],
        n_neurons_plot=2 * n_neurons,
        n_compared=2,
        fraction=1.0,
        random_seed=None,
        title=f"Teacher vs Trained Student Spike Trains (seed-{example_seed}, noise=0.4)",
        ylabel="Neuron",
        figsize=(16, 8),
    )

    plt.tight_layout()
    plt.show()

In [ ]:
if not (HAS_CONVERGENCE and HAS_INFERENCE):
    print("SKIPPED: need both convergence and inference data")
else:
    correct_student_plot_data = np.load(inference_dir / "plot_data.npz")
    correct_student_spikes = correct_student_plot_data["student_spikes"]

    interleaved_3way, ct_idx_3way = interleave_spike_trains(
        teacher_spikes, student_spikes, correct_student_spikes
    )

    fig = plot_spike_trains(
        spikes=interleaved_3way,
        dt=dt,
        cell_type_indices=ct_idx_3way,
        cell_type_names=["Teacher", "Trained Student", "Correct Student"],
        n_neurons_plot=3 * n_neurons,
        n_compared=3,
        fraction=1.0,
        random_seed=None,
        title=f"Teacher vs Trained vs Correct Student Spike Trains (seed-{example_seed}, noise=0.4)",
        ylabel="Neuron",
        figsize=(18, 10),
        show_spike_histogram=True,
    )

    plt.tight_layout()
    plt.show()